# FlowEdit Colab Notebook (SD3 Euler vs Midpoint)
This notebook is designed to compare **Euler** and **Midpoint** solvers for Stable Diffusion 3 (SD3) FlowEdit directly in Google Colab.

Features:
- Optional Google Drive workspace
- Manual Hugging Face token input
- Pinned dependency versions for better stability

Before running, set runtime to GPU: `Runtime -> Change runtime type -> GPU`.


In [ ]:
# 1) Run configuration (edit this cell first)
USE_DRIVE = False  # True: use Google Drive workspace, False: stay in /content
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/FlowEdit"

# Use your fork URL here
REPO_URL = "https://github.com/Jiaqi-Ye/FlowEdit.git"
REPO_DIR = "/content/FlowEdit"

# Branch to test (midpoint solver branch)
BRANCH = "feature/midpoint-solver"

# SD3 model ID
SD3_MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"

# Experiment name used in output path
EXP_NAME = "FlowEdit_SD3_Colab_Run"

# Manually set your Hugging Face token here
HF_TOKEN = "hf_your_token_here"


In [ ]:
# 2) Check GPU
!nvidia-smi


In [ ]:
# 3) Optional: mount Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted")
else:
    print("Drive mount skipped")


In [ ]:
# 4) Clone repository, checkout branch, and switch to workspace
import os
import shutil
from pathlib import Path

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

if USE_DRIVE:
    Path(DRIVE_PROJECT_DIR).parent.mkdir(parents=True, exist_ok=True)
    if not os.path.exists(DRIVE_PROJECT_DIR):
        shutil.copytree(REPO_DIR, DRIVE_PROJECT_DIR)
    %cd {DRIVE_PROJECT_DIR}
else:
    %cd {REPO_DIR}

!git fetch --all
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!git branch --show-current
!ls


In [ ]:
# 5) Install dependencies (pinned)
!pip -q install --upgrade pip setuptools wheel
!pip -q install torch
!pip -q install diffusers==0.30.1 transformers==4.44.2 accelerate==0.33.0
!pip -q install sentencepiece protobuf safetensors pyyaml huggingface_hub


In [ ]:
# 6) Validate HF token (manual input)
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("Please set HF_TOKEN manually in Cell 1.")

print("HF_TOKEN format looks valid")


In [ ]:
# 7) Login to Hugging Face using manual HF_TOKEN
from huggingface_hub import login

login(token=HF_TOKEN)
print("Hugging Face login successful")


In [ ]:
# 8) Generate SD3 Euler vs Midpoint comparison YAML
import textwrap

exp_yaml_name = "SD3_solver_compare_colab.yaml"
exp_yaml_template = """
- 
  exp_name: "{EXP_NAME}_Euler"
  dataset_yaml: edits.yaml
  model_type: "SD3"
  model_id: "{SD3_MODEL_ID}"
  sampler_type: "FlowEditSD3"
  solver_type: "euler"
  T_steps: 50
  n_avg: 1
  src_guidance_scale: 3.5
  tar_guidance_scale: 13.5
  n_min: 0
  n_max: 33
  seed: 42

- 
  exp_name: "{EXP_NAME}_Midpoint"
  dataset_yaml: edits.yaml
  model_type: "SD3"
  model_id: "{SD3_MODEL_ID}"
  sampler_type: "FlowEditSD3"
  solver_type: "midpoint"
  T_steps: 50
  n_avg: 1
  src_guidance_scale: 3.5
  tar_guidance_scale: 13.5
  n_min: 0
  n_max: 33
  seed: 42
"""
exp_yaml = exp_yaml_template.format(EXP_NAME=EXP_NAME, SD3_MODEL_ID=SD3_MODEL_ID)

with open(exp_yaml_name, "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(exp_yaml).lstrip())

print("Wrote", exp_yaml_name)
!cat {exp_yaml_name}


In [ ]:
# 9) Pre-run file check
import os
required_files = ["run_script.py", "FlowEdit_utils.py", "make_solver_comparison.py", "edits.yaml", exp_yaml_name]
for rf in required_files:
    assert os.path.exists(rf), f"Missing file: {rf}"
print("All required files found.")


In [ ]:
# 10) Run FlowEdit
!python run_script.py --exp_yaml {exp_yaml_name}


In [ ]:
# 11) Build and preview Euler vs Midpoint comparison sheet
import glob
from IPython.display import display
from PIL import Image

comparison_path = "outputs/solver_comparison/sd3_euler_vs_midpoint.png"
!python make_solver_comparison.py --dataset_yaml edits.yaml --euler_exp {EXP_NAME}_Euler --midpoint_exp {EXP_NAME}_Midpoint --out {comparison_path}

paths = sorted(glob.glob(f"outputs/{EXP_NAME}_*/SD3/**/output_*.png", recursive=True))
print(f"Found {len(paths)} solver output images")
print("Comparison sheet:", comparison_path)
display(Image.open(comparison_path))

for p in paths[:6]:
    print(p)
    display(Image.open(p))


## Troubleshooting
- `401/403`: token permission issue. Re-check `HF_TOKEN` in Cell 1.
- `CUDA out of memory`: reduce image size and close other GPU sessions.
- Slow first run: initial model download can take a while; cached runs are faster.
